<a href="https://colab.research.google.com/github/Maryam-Taherzadeh/Computational-Drug-Discovery/blob/main/project-01-predict-ic50-mdm2-p53/notebooks/mdm2_part1_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Computational Drug Discovery – MDM2 Inhibitor Prediction

Author: Maryam Taherzadeh  
Project: Computational Drug Discovery – MDM2  

## Part 1: Bioactivity Data Collection & Preprocessing

---

## 📌 Project Overview

In this notebook, we develop a real-world data science project focused on predicting the bioactivity (IC50) of compounds targeting the **MDM2 protein**, a key regulator of the tumor suppressor p53 in cancer progression.

The goal of this part is to retrieve and preprocess bioactivity data from the ChEMBL database, preparing it for machine learning modeling.

---

## 🎯 Objectives

- Retrieve bioactivity data for the MDM2 target from ChEMBL  
- Filter relevant activity data (IC50 values)  
- Handle missing values  
- Clean and standardize the dataset  
- Label compounds as:
  - **Active**
  - **Intermediate**
  - **Inactive**  
- Prepare a structured dataset for machine learning  

---

## ⚙️ Workflow

In this notebook, we perform the following steps:

1. Access ChEMBL database  
2. Extract bioactivity data for the MDM2 protein (CHEMBL5023)  
3. Filter IC50 values  
4. Handle missing data  
5. Label compounds based on activity thresholds  
6. Save processed dataset  

---

## 🧪 Bioactivity Labeling Criteria

- **Active**: IC50 ≤ 1000 nM  
- **Intermediate**: 1000 nM < IC50 < 10000 nM  
- **Inactive**: IC50 ≥ 10000 nM  

---

## 📊 Applications

This workflow is widely used in:

- Drug discovery  
- QSAR modeling  
- Virtual screening  
- Lead optimization  

---

## 🚀 Next Step

In the next part, we will:

- Convert SMILES into numerical features using RDKit  
- Calculate molecular descriptors (e.g., Lipinski properties)  
- Prepare data for machine learning models  

### **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications.
[Data as of March 25, 2020; ChEMBL version 26].





## **Installing libraries**
###Install ChEMBL Web Resource Client

In this step, we install the ChEMBL web resource client, which allows us to retrieve bioactivity data directly from the ChEMBL.

This client provides a convenient interface to access:

* molecular data
* bioactivity data (e.g., IC50)
* target information

In [86]:
! pip install chembl_webresource_client

## **Importing libraries**

In [2]:
# Import necessary libraries

import pandas as pd
!pip install chembl_webresource_client --upgrade
from chembl_webresource_client.new_client import new_client


## **Search for Target protein**

## **Search for MDM2**

In [3]:
target = new_client.target
target_query = target.search('MDM2')
targets = pd.DataFrame.from_dict(target_query)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

targets




,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,E3 ubiquitin-protein ligase Mdm2,18.0,False,CHEMBL5023,"[{'accession': 'Q00987', 'component_descriptio...",SINGLE PROTEIN,9606
1,[],Canis lupus familiaris,E3 ubiquitin-protein ligase Mdm2,18.0,False,CHEMBL3600278,"[{'accession': 'P56950', 'component_descriptio...",SINGLE PROTEIN,9615
2,[],Mus musculus,E3 ubiquitin-protein ligase Mdm2,18.0,False,CHEMBL3600279,"[{'accession': 'P23804', 'component_descriptio...",SINGLE PROTEIN,10090
3,[],Homo sapiens,Protein cereblon/E3 ubiquitin-protein ligase Mdm2,18.0,False,CHEMBL4523702,"[{'accession': 'Q00987', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
4,[],Homo sapiens,Tumour suppressor p53/oncoprotein Mdm2,17.0,False,CHEMBL1907611,"[{'accession': 'P04637', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
5,[],Homo sapiens,Casein kinase I/MDM2,17.0,False,CHEMBL3038493,"[{'accession': 'Q00987', 'component_descriptio...",PROTEIN COMPLEX,9606
6,[],Homo sapiens,MDM2-MDMX,17.0,False,CHEMBL4106123,"[{'accession': 'Q00987', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
7,[],Homo sapiens,E3 ubiquitin-protein ligase Mdm2/CDK6,17.0,False,CHEMBL4523718,"[{'accession': 'Q00987', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
8,[],Homo sapiens,BRD4/E3 ubiquitin-protein ligase Mdm2,17.0,False,CHEMBL4523719,"[{'accession': 'Q00987', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
9,[],Homo sapiens,Protein cereblon/Tumour suppressor p53/oncopro...,17.0,False,CHEMBL4879538,"[{'accession': 'P04637', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606


# Select and retrieve bioactivity data for the MDM2 target
---



We assign the first entry (which corresponds to the human MDM2 protein target) to the **selected_target** variable.

In [4]:
selected_target = targets[
    (targets.target_chembl_id == 'CHEMBL5023') &
    (targets.organism == 'Homo sapiens')
].target_chembl_id.values[0]

selected_target

'CHEMBL5023'

In this step, we retrieve bioactivity data for the human MDM2 protein (CHEMBL5023) from the ChEMBL database, focusing on compounds with experimentally measured IC50 values reported in nanomolar (nM) units. These values will be used to quantify inhibitory activity and serve as the target variable for downstream machine learning modeling.

In [10]:
activity = new_client.activity

res = activity.filter(
    target_chembl_id=selected_target,
    standard_type="IC50"
).only([
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_value',
    'standard_type'
])

df = pd.DataFrame(res)

In [11]:
df = pd.DataFrame.from_dict(res)


In [12]:
df.head()


,canonical_smiles,molecule_chembl_id,standard_type,standard_value,type,value
0,O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[...,CHEMBL178578,IC50,1700.0,IC50,1.7
1,O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(...,CHEMBL361103,IC50,200.0,IC50,0.2
2,C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H]...,CHEMBL182051,IC50,30000.0,IC50,30.0
3,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCC...,CHEMBL360540,IC50,13000.0,IC50,13.0
4,C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CC...,CHEMBL427316,IC50,3600.0,IC50,3.6


In [13]:
#It returns all different activity types in your dataset.
df.standard_type.unique()

array(['IC50'], dtype=object)

In [14]:
df.shape

(5820, 6)

Finally we will save the resulting bioactivity data to a CSV file **mdm2_data.csv**.

In [64]:
# save it locally
df.to_csv('mdm2_data.csv', index=False)

## **Copying files to Google Drive**

Firstly, we need to mount the Google Drive into Colab so that we can have access to our Google adrive from within Colab.

In [65]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)


Mounted at /content/gdrive/


Next, we create a **data** folder in our **Colab Notebooks** folder on Google Drive.

In [66]:
# create folder safely
!mkdir -p "/content/gdrive/My Drive/Colab Notebooks/data2"

# copy file into folder
!cp mdm2_data.csv "/content/gdrive/My Drive/Colab Notebooks/data2"

# list files
!ls "/content/gdrive/My Drive/Colab Notebooks/data2"

# show detail info
!ls -l "/content/gdrive/My Drive/Colab Notebooks/data2"

# check current directory
!pwd

mdm2_data.csv
total 674
-rw------- 1 root root 689640 Apr 21 23:10 mdm2_data.csv
/content


Let's see the CSV files that we have so far.

In [67]:

!ls

gdrive	mdm2_data.csv  mdm2_preprocessed_data.csv  sample_data


Taking a glimpse of the **mdm2_data.csv.csv** file that we've just created.

In [68]:
! head mdm2_data.csv


canonical_smiles,molecule_chembl_id,standard_type,standard_value,type,value
O=C(O)[C@H](c1ccccc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(C(F)(F)F)cc1,CHEMBL178578,IC50,1700.0,IC50,1.7
O=C(O)[C@H](c1ccc(Cl)cc1)N1C(=O)c2cc(I)ccc2NC(=O)[C@@H]1c1ccc(Cl)cc1,CHEMBL361103,IC50,200.0,IC50,0.2
C[C@H](c1ccc(Cl)cc1)N(C(=O)/C=C/c1ccccc1)[C@H](C(N)=O)c1ccc(Cl)cc1,CHEMBL182051,IC50,30000.0,IC50,30.0
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1,CHEMBL360540,IC50,13000.0,IC50,13.0
C[C@H](c1ccc(Cl)cc1)N1C(=O)C=C(c2ccccc2Br)N(CCCCC(=O)O)C(=O)[C@@H]1c1ccc(Cl)cc1,CHEMBL427316,IC50,3600.0,IC50,3.6
O=C(O)C(c1ccccc1)N(Cc1ccc(C(F)(F)F)cc1)C(=O)c1cccc(I)c1,CHEMBL425528,IC50,15000.0,IC50,15.0
C=C(C(=O)O)C(CCCCCC)C(=O)O,CHEMBL199341,IC50,50.0,IC50,50.0
COc1ccc(C2=N[C@@H](c3ccc(Cl)cc3)[C@@H](c3ccc(Cl)cc3)N2C(=O)N2CCNC(=O)C2)c(OC(C)C)c1,CHEMBL191334,IC50,1500.0,IC50,1500.0
CC(C)(C)C[C@H]1N[C@@H](C(=O)NCCN2CCOCC2)[C@H](c2cccc(Cl)c2F)[C@@]12C(=O)Nc1cc(Cl)ccc12,CHEMBL379173,IC50,280.0,IC

## **Handling missing data**
If any compounds has missing value for the **standard_value** column then drop it

In [69]:
df.columns


Index(['canonical_smiles', 'molecule_chembl_id', 'standard_type',
       'standard_value', 'type', 'value'],
      dtype='object')

In [70]:
df.standard_value.isna().sum()

np.int64(0)

In [71]:
df.canonical_smiles.isna().sum()


np.int64(0)

In [72]:
df = df.dropna(subset=['canonical_smiles', 'standard_value'])

In [73]:
df.shape

(5792, 6)

## **Data pre-processing of the bioactivity data**

### **Labeling compounds as either being active, inactive or intermediate**
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be **active** while those greater than 10,000 nM will be considered to be **inactive**. As for those values in between 1,000 and 10,000 nM will be referred to as **intermediate**.

In [74]:
bioactivity_class = []
for i in df.standard_value:
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")


### **Iterate the *molecule_chembl_id* to a list**

In [75]:
mol_cid = []
for i in df.molecule_chembl_id:
  mol_cid.append(i)

### **Iterate *canonical_smiles* to a list**

In [76]:
canonical_smiles = []
for i in df.canonical_smiles:
  canonical_smiles.append(i)

### **Iterate *standard_value* to a list**

In [77]:
standard_value = []
for i in df.standard_value:
  standard_value.append(i)

### **Combine the 4 lists into a dataframe**

In [78]:
data_tuples = list(zip(mol_cid, canonical_smiles, bioactivity_class, standard_value))

df2 = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'standard_value'])

Saves dataframe to CSV *file*

In [79]:
df2.to_csv('mdm2_preprocessed_data.csv', index=False)
df2.shape

(5792, 4)

In [80]:
! ls -l


total 1316
drwx------ 5 root root   4096 Apr 21 23:10 gdrive
-rw-r--r-- 1 root root 689640 Apr 21 23:09 mdm2_data.csv
-rw-r--r-- 1 root root 645351 Apr 21 23:10 mdm2_preprocessed_data.csv
drwxr-xr-x 1 root root   4096 Apr 16 13:28 sample_data


Let's copy to the Google Drive

In [84]:
!mkdir -p "/content/gdrive/My Drive/Colab Notebooks/data2"
!cp mdm2_*.csv "/content/gdrive/My Drive/Colab Notebooks/data2/"


In [85]:
! ls "/content/gdrive/My Drive/Colab Notebooks/data2"



mdm2_data.csv  mdm2_preprocessed_data.csv


---